# 🏭 Tennessee Eastman Process (TEP) - 3-Pillar Heterogeneous Model Training
### Production GPU Training Pipeline for Predictive Maintenance & 21-Class Fault Isolation

This notebook trains the complete 3-pillar industrial model ensemble for TEP:
- **Pillar 1:** XGBoost (Binary Fault Detection + 21-Class Multiclass Isolation with GPU Hist)
- **Pillar 2:** PyTorch LSTM-FCN (Temporal Sequence Model with Squeeze-and-Excitation + CUDA AMP fp16)
- **Pillar 3:** PyTorch Dense Autoencoder (Unsupervised Steady-State Reconstruction)
- **Stacking Meta-Learner:** Calibrated Logistic Regression unified risk scoring

**Output:** Produces `models/tep_artifacts.zip` containing `bundle.joblib`, `lstm_fcn.pt`, `autoencoder.pt`, `metrics.json`, and `replay_sample.parquet` ready to drop into the local/Docker app.

### Step 1: Verify GPU Environment (Recommended: T4 or A100 GPU)

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")

### Step 2: Install Dependencies

In [ ]:
!pip install -q xgboost scikit-learn pandas numpy joblib kagglehub pyarrow

### Step 3: Run 3-Pillar Training Pipeline
Choose your dataset source:
- **Option A (Real TEP Kaggle Data - Recommended):** `--dataset kaggle_dat` downloads the benchmark `.dat` files via kagglehub without requiring credentials.
- **Option B (Real TEP CSV Files):** Place extended CSV files into `./data/` and run `--dataset csv --max-runs-per-fault 40`.
- **Option C (Synthetic Simulation):** `--dataset synthetic --max-runs-per-fault 15`.

In [ ]:
# Real TEP benchmark training on GPU with AMP fp16
!python -m ml.train_colab --dataset kaggle_dat --output-dir ./models

### Step 4: Inspect Benchmark Metrics

In [ ]:
import json
with open('./models/metrics.json') as f:
    metrics = json.load(f)
print(json.dumps(metrics, indent=2))

### Step 5: Download Production Artifacts
Download `tep_artifacts.zip` and extract directly into your project's `/models` folder (or save to Google Drive).

In [ ]:
from google.colab import files
files.download('./models/tep_artifacts.zip')

### (Optional) Backup to Google Drive

In [ ]:
# Uncomment to save directly to Google Drive
# from google.colab import drive
# import shutil, os
# drive.mount('/content/drive')
# os.makedirs('/content/drive/MyDrive/TEP_Models', exist_ok=True)
# shutil.copy('./models/tep_artifacts.zip', '/content/drive/MyDrive/TEP_Models/tep_artifacts.zip')
# print('Saved to Google Drive!')